# 01 · Setup ADLS — Squad 3 | Henrique Ficht
Valida a conexão com o Azure Data Lake Gen2 e lista os arquivos
disponíveis no container `batch-data` para a Squad 3.

In [0]:
%pip install python-dotenv azure-storage-blob azure-identity --quiet
dbutils.library.restartPython()

In [0]:
import os
from dotenv import load_dotenv

ENV_PATH = "/Workspace/Users/ik.kukoo@gmail.com/.env"
load_dotenv(ENV_PATH, override=True)

def _require(key):
    value = os.getenv(key)
    if not value:
        raise EnvironmentError(f"Variável '{key}' não encontrada em {ENV_PATH}")
    return value

STORAGE_ACCOUNT = "internshipdatalake"
CONTAINER       = "raw"

adls_options = {
    f"fs.azure.account.auth.type.{STORAGE_ACCOUNT}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{STORAGE_ACCOUNT}.dfs.core.windows.net":
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{STORAGE_ACCOUNT}.dfs.core.windows.net":
        _require("ADLS_CLIENT_ID"),
    f"fs.azure.account.oauth2.client.secret.{STORAGE_ACCOUNT}.dfs.core.windows.net":
        _require("ADLS_CLIENT_SECRET"),
    f"fs.azure.account.oauth2.client.endpoint.{STORAGE_ACCOUNT}.dfs.core.windows.net":
        f"https://login.microsoftonline.com/{_require('ADLS_TENANT_ID')}/oauth2/token",
}

def adls_path(subfolder=""):
    base = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/batch-data"
    return f"{base}/{subfolder}" if subfolder else base

print("✅ Config carregado.")
print(f"   ADLS → {adls_path()}")

In [0]:
# Listar raiz do container 
root_path = adls_path()

files_root = (
    spark.read
    .format("binaryFile")
    .options(**adls_options)
    .load(root_path)
    .select("path")
)

print(f"📂 Conteúdo de: {root_path}\n")
files_root.show(50, truncate=False)

In [0]:
# Localizar os arquivos da Squad 3
TARGET_FILES = ["food_estoque_lojas", "food_avaliacoes_produto"]

all_paths = [row.path for row in files_root.collect()]

found = {name: [p for p in all_paths if name in p] for name in TARGET_FILES}

print("🔍 Arquivos encontrados:\n")
for name, paths in found.items():
    if paths:
        for p in paths:
            print(f"  ✅ {name}: {p}")
    else:
        print(f"  ❌ {name}: NÃO encontrado — verifique o caminho no container")

In [0]:
# Leitura de Amostra
ESTOQUE_PATH    = adls_path("food_estoque_lojas.csv")
AVALIACOES_PATH = adls_path("food_avaliacoes_produto.csv")

for label, path in [("food_estoque_lojas", ESTOQUE_PATH),
                    ("food_avaliacoes_produto", AVALIACOES_PATH)]:
    try:
        df_sample = (
            spark.read
            .options(**adls_options)
            .option("header", "true")
            .option("inferSchema", "true")
            .csv(path)
        )
        print(f"\n✅ {label} — {df_sample.count()} linhas | {len(df_sample.columns)} colunas")
        df_sample.printSchema()
    except Exception as e:
        print(f"\n❌ Erro ao ler {label}: {e}")